In [19]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver , InMemorySaver

In [20]:
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [21]:
from langgraph.graph.message import add_messages
class ChatBot(TypedDict):
    messege: Annotated[list[BaseMessage], add_messages]

In [22]:
def chat_node(state: ChatBot) -> ChatBot:

    messe = state['messege']


    model_response = model.invoke(messe)

    return {'messege':[model_response]}

In [23]:
checkPointer = InMemorySaver()
graph = StateGraph(ChatBot)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)


In [24]:
thread_id = '1'

while True:
    user_message = input("You: ").strip()
    print("User:", user_message)       

    if user_message.lower() in {"exit", "quit"}:
        print("Bot: Bye!")
        break
    config = {"configurable": {"thread_id": "1"}}
    response = workflow.invoke({
        "messege": [HumanMessage(content=user_message)]
    }, config=config)["messege"][-1].text

    print("Bot:", response)

User: 10 + 5
Bot: 15
User: add 2
Bot: 17
User: exit
Bot: Bye!


In [18]:
state = workflow.get_state(
    config={"configurable": {"thread_id": "1"}}
)
print(state)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())


In [17]:
list[workflow.get_state_history(config)]

list[<generator object Pregel.get_state_history at 0x1120dfca0>]